# Notebook 06 — Governance Agent Evaluation
This notebook loads the trained Risk Assessment Agent's scores and evaluates the deterministic rule engine (Governance Agent) based on grounded policy rules from OWASP LLM Top 10 2025 and NIST AI RMF.

### Research & Governance Design:
1. **Explainable Auditable Policy:** All decisions must trace back to a specific named rule (e.g. G1–G10) citing a concrete regulatory policy reference.
2. **Defensive Depth (Three-tier system):**
   - **BLOCK:** Rejects documents immediately (hard risk boundary).
   - **SANITIZE:** Redacts dangerous segments before forwarding.
   - **ALLOW:** Permits the document to proceed to the LLM.
3. **Threshold Tuning:** Tune critical risk thresholds strictly on the validation set to keep FNR < 5%.

In [ ]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, f1_score

COLAB_BASE = "/content/drive/MyDrive/PAS"
if os.path.exists(COLAB_BASE):
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    BASE = COLAB_BASE
else:
    if os.path.isdir("NOTEBOOKS"):
        BASE = str(Path(".").resolve())
    elif os.path.isdir("../NOTEBOOKS"):
        BASE = str(Path("..").resolve())
    else:
        BASE = str(Path(".").resolve().parent)
import sys
sys.path.insert(0, os.path.join(BASE, "AGENTS"))
COLAB_BASE = "/content/drive/MyDrive/PAS"
if os.path.exists(COLAB_BASE):
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    BASE = COLAB_BASE
else:
    if os.path.isdir("NOTEBOOKS"):
        BASE = str(Path(".").resolve())
    elif os.path.isdir("../NOTEBOOKS"):
        BASE = str(Path("..").resolve())
    else:
        BASE = str(Path(".").resolve().parent)
import sys
sys.path.insert(0, os.path.join(BASE, "AGENTS"))
COLAB_BASE = "/content/drive/MyDrive/PAS"
if os.path.exists(COLAB_BASE):
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    BASE = COLAB_BASE
else:
    if os.path.isdir("NOTEBOOKS"):
        BASE = str(Path(".").resolve())
    elif os.path.isdir("../NOTEBOOKS"):
        BASE = str(Path("..").resolve())
    else:
        BASE = str(Path(".").resolve().parent)
import sys
sys.path.insert(0, os.path.join(BASE, "AGENTS"))
COLAB_BASE = "/content/drive/MyDrive/PAS"
if os.path.exists(COLAB_BASE):
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    BASE = COLAB_BASE
else:
    if os.path.isdir("NOTEBOOKS"):
        BASE = str(Path(".").resolve())
    elif os.path.isdir("../NOTEBOOKS"):
        BASE = str(Path("..").resolve())
    else:
        BASE = str(Path(".").resolve().parent)
import sys
sys.path.insert(0, os.path.join(BASE, "AGENTS"))
CACHE_DIR   = os.path.join(BASE, "DATASETS", "EXPERIMENT_CACHE")
if not os.path.exists(CACHE_DIR):
    CACHE_DIR = os.path.join(BASE, "EXPERIMENT_CACHE")

MODELS_DIR  = os.path.join(BASE, "MODELS")
RESULTS_DIR = os.path.join(BASE, "RESULTS")
METRICS_DIR = os.path.join(RESULTS_DIR, "metrics"); os.makedirs(METRICS_DIR, exist_ok=True)
RULES_PATH  = os.path.join(BASE, "RULES", "governance_rules.json")

from governance_agent import GovernanceAgent

gov = GovernanceAgent(rules_path=RULES_PATH)
rules = gov.list_rules()
rules_df = pd.DataFrame(rules)
print(f"Loaded {len(rules)} governance rules:")
display(rules_df)

In [ ]:
features_path = os.path.join(CACHE_DIR, "features.parquet")
risk_path = os.path.join(CACHE_DIR, "risk_scores.csv")

assert os.path.exists(features_path), f"features.parquet not found: {features_path}"
assert os.path.exists(risk_path), f"risk_scores.csv not found: {risk_path}"

features_df = pd.read_parquet(features_path)
risk_df = pd.read_csv(risk_path)[["sample_id", "risk_score", "risk_level"]]

# Merge risk score into features dataframe
df = features_df.merge(risk_df, on="sample_id", how="inner")
print(f"Merged dataframe shape: {df.shape}")

val_df = df[df["split"] == "val"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

print(f"Validation set size: {len(val_df):,}")
print(f"Test set size: {len(test_df):,}")

In [ ]:
# Tune optimal risk_score threshold for G1 (unconditional block) on the validation set.
# Ground truth label: 'label_int' (1=malicious, 0=benign)

best_th = 0.80
best_f1 = 0.0
fnr_at_best = 1.0

# Search space for threshold
threshold_candidates = np.linspace(0.50, 0.90, 41)

print("Tuning G1 risk threshold on validation set...")
for th in threshold_candidates:
    decisions = []
    for _, row in val_df.iterrows():
        r_score = row["risk_score"]
        if r_score >= th:
            dec = "BLOCK"
        else:
            d_obj = gov.evaluate(
                risk_score=r_score,
                vision_score=row.get("vision_score", 0.0),
                hidden_text_detected=row.get("hidden_text_score", 0.0) >= 0.5,
                keyword_density=row.get("keyword_density", 0.0),
                footer_density=row.get("footer_text_density", 0.0),
                watermark_score=row.get("watermark_score", 0.0),
                prompt_score=row["malicious_probability"],
                severity=row.get("severity", "medium") if "severity" in row else "medium"
            )
            dec = d_obj.decision
        decisions.append(1 if dec in ["BLOCK", "SANITIZE"] else 0)
        
    y_val_pred = np.array(decisions)
    y_val_true = val_df["label_int"].values
    
    val_f1 = f1_score(y_val_true, y_val_pred, average="macro")
    val_fnr = ((y_val_true == 1) & (y_val_pred == 0)).sum() / max(1, (y_val_true == 1).sum())
    
    if val_fnr <= 0.05: # safety-critical constraint: FNR < 5%
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_th = th
            fnr_at_best = val_fnr

print(f"\n[Tuned Optimal Threshold] G1 Risk Score: {best_th:.2f}")
print(f"Validation F1-Macro: {best_f1:.4%}")
print(f"Validation FNR: {fnr_at_best:.4%}")

# Update G1 threshold in rules for future evaluations
for rule in gov._rules:
    if rule["id"] == "G1":
        rule["condition"]["risk_score"]["gte"] = float(best_th)
        print("Updated G1 threshold in loaded GovernanceAgent instance.")

In [ ]:
# Apply governance rules to full test set using the tuned GovernanceAgent
test_decisions = []
for _, row in test_df.iterrows():
    d = gov.evaluate(
        risk_score=row["risk_score"],
        vision_score=row.get("vision_score", 0.0),
        hidden_text_detected=row.get("hidden_text_score", 0.0) >= 0.5,
        keyword_density=row.get("keyword_density", 0.0),
        footer_density=row.get("footer_text_density", 0.0),
        watermark_score=row.get("watermark_score", 0.0),
        prompt_score=row["malicious_probability"],
        severity=row.get("severity", "medium") if "severity" in row else "medium"
    )
    test_decisions.append({
        "sample_id": row["sample_id"],
        "decision": d.decision,
        "rule_id": d.rule_id,
        "rule_name": d.rule_name,
        "severity": d.severity,
        "policy_ref": d.policy_ref,
        "reason": d.reason
    })

dec_df = pd.DataFrame(test_decisions)
test_results = test_df.merge(dec_df, on="sample_id", how="inner")
print(f"Applied rules to test set. Decision distribution:\n{test_results['decision'].value_counts()}")

In [ ]:
# Plot decision distribution by ground truth label
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, label_val, label_name in zip(axes, [0, 1], ["benign", "malicious"]):
    subset = test_results[test_results["label_int"] == label_val]
    counts = subset["decision"].value_counts().reindex(["ALLOW", "SANITIZE", "BLOCK"], fill_value=0)
    colors = ["#2ecc71", "#f39c12", "#e74c3c"]
    ax.bar(counts.index, counts.values, color=colors, edgecolor="black", linewidth=1.2)
    ax.set_title(f"{label_name.upper()} — Decision Distribution", fontweight="bold")
    ax.set_ylabel("Count")
    for bar, v in zip(ax.patches, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, v + max(5, v*0.02), f"{v:,}",
                ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.suptitle("Governance Agent Decision Distribution by Ground Truth Label", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(METRICS_DIR, "06_governance_decisions.png"), bbox_inches="tight", dpi=150)
plt.show()

# Governance Confusion Matrix (mapping ALLOW -> 0, BLOCK/SANITIZE -> 1)
y_pred_bin = test_results["decision"].isin(["BLOCK", "SANITIZE"]).astype(int)
y_true_bin = test_results["label_int"].values

cm = confusion_matrix(y_true_bin, y_pred_bin)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
            xticklabels=["ALLOW", "BLOCK/SANITIZE"], yticklabels=["Benign", "Malicious"])
ax.set_title("Governance Decision Confusion Matrix", fontweight="bold")
ax.set_ylabel("Ground Truth Label")
ax.set_xlabel("Governance Action")
plt.tight_layout()
plt.savefig(os.path.join(METRICS_DIR, "06_governance_confusion.png"), bbox_inches="tight", dpi=150)
plt.show()

fnr = ((y_true_bin == 1) & (y_pred_bin == 0)).sum() / max(1, (y_true_bin == 1).sum())
fpr = ((y_true_bin == 0) & (y_pred_bin == 1)).sum() / max(1, (y_true_bin == 0).sum())
print(f"Governance FNR (Malicious allowed): {fnr:.4%}")
print(f"Governance FPR (Benign blocked/sanitized): {fpr:.4%}")

In [ ]:
# Breakdown of decisions per attack family
if "attack_family" in test_results.columns:
    family_decisions = test_results[test_results["label_int"] == 1].groupby("attack_family")["decision"].value_counts().unstack(fill_value=0)
    for col in ["ALLOW", "SANITIZE", "BLOCK"]:
        if col not in family_decisions.columns:
            family_decisions[col] = 0
    family_decisions["Total"] = family_decisions.sum(axis=1)
    family_decisions["Detection Rate (BLOCK/SANITIZE)"] = (family_decisions["BLOCK"] + family_decisions["SANITIZE"]) / family_decisions["Total"]
    family_decisions = family_decisions.sort_values(by="Detection Rate (BLOCK/SANITIZE)", ascending=False)
    print("\n--- Governance Performance by Attack Family ---")
    display(family_decisions.round(4))

In [ ]:
# Save decisions
gov_decisions_path = os.path.join(CACHE_DIR, "governance_decisions.csv")
test_results[["sample_id", "decision", "rule_id", "rule_name", "severity", "policy_ref", "reason"]].to_csv(gov_decisions_path, index=False)
print(f"Governance decisions saved to: {gov_decisions_path}")